In [1]:
import os
import psycopg2

conn = psycopg2.connect(
    host=os.getenv("PGHOST"),
    port=os.getenv("PGPORT"),
    user=os.getenv("PGUSER"),
    password=os.getenv("PGPASSWORD"),
    dbname="cinema"
)

cur = conn.cursor()

print("Connected to:", conn.get_dsn_parameters()["dbname"])

Connected to: cinema


In [2]:
cur.execute("""
CREATE OR REPLACE VIEW schema_clean.vw_people AS
SELECT DISTINCT ON (nconst)
    SUBSTRING(nconst FROM 3)::INTEGER AS person_id,
    TRIM("primaryName") AS person_name
FROM schema_temporary.name_basics
WHERE nconst ~ '^nm[0-9]{1,9}$'
  AND "primaryName" IS NOT NULL
  AND TRIM("primaryName") NOT IN ('', '\\N')
ORDER BY nconst, TRIM("primaryName");
""")

conn.commit()

print("People view created successfully!")

People view created successfully!


In [3]:
import pandas as pd

cur.execute("""
SELECT *
FROM schema_clean.vw_people
LIMIT 10;
""")

columns = [desc[0] for desc in cur.description]
rows = cur.fetchall()

df_people = pd.DataFrame(rows, columns=columns)

display(df_people)

,person_id,person_name
0,1,Fred Astaire
1,2,Lauren Bacall
2,3,Brigitte Bardot
3,4,John Belushi
4,5,Ingmar Bergman
5,6,Ingrid Bergman
6,7,Humphrey Bogart
7,8,Marlon Brando
8,9,Richard Burton
9,10,James Cagney


In [4]:
cur.execute("""
SELECT *
FROM schema_clean.vw_people
LIMIT 5;
""")

for row in cur.fetchall():
    print(row)

(1, 'Fred Astaire')
(2, 'Lauren Bacall')
(3, 'Brigitte Bardot')
(4, 'John Belushi')
(5, 'Ingmar Bergman')


In [7]:
cur.execute(r"""
CREATE OR REPLACE VIEW schema_clean.vw_movie_cast AS
SELECT DISTINCT
    SUBSTRING(tconst FROM 3)::INTEGER AS movie_id,
    SUBSTRING(nconst FROM 3)::INTEGER AS person_id,
    NULLIF(TRIM(characters), '\N') AS character_name
FROM schema_temporary.title_principals
WHERE category IN ('actor', 'actress')
  AND tconst ~ '^tt[0-9]{1,9}$'
  AND nconst ~ '^nm[0-9]{1,9}$';
""")

conn.commit()
print("Movie cast view created!")

Movie cast view created!


In [8]:
cur.execute("""
SELECT *
FROM schema_clean.vw_movie_cast
LIMIT 5;
""")

for row in cur.fetchall():
    print(row)

(5, 443482, '["Blacksmith"]')
(5, 653042, '["Assistant"]')
(7, 179163, None)
(7, 183947, None)
(8, 653028, '["Sneezing Man"]')


In [9]:
cur.execute(r"""
CREATE OR REPLACE VIEW schema_clean.vw_movie_crew AS
SELECT DISTINCT
    SUBSTRING(tconst FROM 3)::INTEGER AS movie_id,
    SUBSTRING(nconst FROM 3)::INTEGER AS person_id,

    COALESCE(
        NULLIF(TRIM(job), '\N'),
        category
    ) AS job

FROM schema_temporary.title_principals
WHERE category NOT IN ('actor', 'actress')
  AND tconst ~ '^tt[0-9]{1,9}$'
  AND nconst ~ '^nm[0-9]{1,9}$';
""")

conn.commit()
print("Movie crew view created!")

Movie crew view created!


In [10]:
cur.execute("""
SELECT *
FROM schema_clean.vw_movie_crew
LIMIT 5;
""")

for row in cur.fetchall():
    print(row)

(1, 5690, 'director')
(1, 5690, 'producer')
(1, 374658, 'director of photography')
(1, 1588970, 'self')
(2, 721526, 'director')


In [11]:
cur.execute(r"""
CREATE OR REPLACE VIEW schema_clean.vw_ratings AS
SELECT DISTINCT
    SUBSTRING(tconst FROM 3)::INTEGER AS movie_id,

    ("averageRating"::NUMERIC * 10)::INTEGER
        AS rating_value,

    'IMDb'::TEXT AS rating_source

FROM schema_temporary.title_ratings
WHERE tconst ~ '^tt[0-9]{1,9}$'
  AND "averageRating" ~ '^[0-9]+(\.[0-9]+)?$'
  AND "averageRating"::NUMERIC BETWEEN 0 AND 10;
""")

conn.commit()
print("Ratings view created!")

Ratings view created!


In [12]:
cur.execute("""
SELECT *
FROM schema_clean.vw_ratings
LIMIT 5;
""")

for row in cur.fetchall():
    print(row)

(1, 57, 'IMDb')
(2, 54, 'IMDb')
(3, 64, 'IMDb')
(4, 50, 'IMDb')
(5, 62, 'IMDb')


In [13]:
cur.execute("""
SELECT table_name
FROM information_schema.views
WHERE table_schema = 'schema_clean'
ORDER BY table_name;
""")

print("Views created:")
for row in cur.fetchall():
    print(row[0])

Views created:
vw_movie_cast
vw_movie_crew
vw_people
vw_ratings


In [14]:
cur.execute("""
CREATE OR REPLACE VIEW schema_clean.vw_movies AS
SELECT
    movie_id,
    imdb_id,
    title,
    release_date,
    release_year,
    runtime
FROM schema_clean.movies;
""")

conn.commit()
print("Movies view created successfully!")

Movies view created successfully!


In [15]:
cur.execute("""
CREATE OR REPLACE VIEW schema_clean.vw_genres AS
SELECT
    genre_id,
    genre_name
FROM schema_clean.genres;
""")

conn.commit()
print("Genres view created successfully!")

Genres view created successfully!


In [16]:
cur.execute("""
CREATE OR REPLACE VIEW schema_clean.vw_movie_genres AS
SELECT
    movie_id,
    genre_id
FROM schema_clean.movie_genres;
""")

conn.commit()
print("Movie-genres view created successfully!")

Movie-genres view created successfully!


In [17]:
cur.execute("""
SELECT table_name
FROM information_schema.views
WHERE table_schema = 'schema_clean'
ORDER BY table_name;
""")

print("All views:")
for row in cur.fetchall():
    print(row[0])

All views:
vw_genres
vw_movie_cast
vw_movie_crew
vw_movie_genres
vw_movies
vw_people
vw_ratings


In [20]:
cur.execute("""
SELECT
    table_name
FROM information_schema.tables
WHERE table_schema = 'public'
  AND table_type = 'BASE TABLE'
ORDER BY table_name;
""")

print("Final public tables:")
for row in cur.fetchall():
    print(row[0])

Final public tables:
genres
movie_cast
movie_crew
movie_genres
movie_studios
movies
people
ratings
reviews
studios


In [21]:
cur.execute("""
SELECT
    relname AS table_name,
    n_live_tup AS estimated_rows
FROM pg_stat_user_tables
WHERE schemaname = 'public'
ORDER BY relname;
""")

for row in cur.fetchall():
    print(row)

('genres', 0)
('movie_cast', 0)
('movie_crew', 0)
('movie_genres', 0)
('movie_studios', 0)
('movies', 0)
('people', 0)
('ratings', 0)
('reviews', 0)
('studios', 0)


In [22]:
cur.execute("""
SELECT
    EXISTS (SELECT 1 FROM public.movies),
    EXISTS (SELECT 1 FROM public.genres),
    EXISTS (SELECT 1 FROM public.people),
    EXISTS (SELECT 1 FROM public.movie_cast),
    EXISTS (SELECT 1 FROM public.movie_crew),
    EXISTS (SELECT 1 FROM public.ratings);
""")

print(cur.fetchone())

(False, False, False, False, False, False)


In [23]:
cur.execute("""
SELECT
    table_name,
    column_name,
    data_type
FROM information_schema.columns
WHERE table_schema = 'public'
ORDER BY table_name, ordinal_position;
""")

for row in cur.fetchall():
    print(row)

('genres', 'genre_id', 'integer')
('genres', 'genre_name', 'text')
('movie_cast', 'movie_id', 'integer')
('movie_cast', 'person_id', 'integer')
('movie_cast', 'character_name', 'text')
('movie_crew', 'movie_id', 'integer')
('movie_crew', 'person_id', 'integer')
('movie_crew', 'job', 'text')
('movie_genres', 'movie_id', 'integer')
('movie_genres', 'genre_id', 'integer')
('movie_studios', 'movie_id', 'integer')
('movie_studios', 'studio_id', 'integer')
('movies', 'movie_id', 'integer')
('movies', 'title', 'text')
('movies', 'release_date', 'date')
('movies', 'runtime', 'integer')
('people', 'person_id', 'integer')
('people', 'name', 'text')
('ratings', 'rating_id', 'integer')
('ratings', 'movie_id', 'integer')
('ratings', 'rating_source', 'text')
('ratings', 'rating_value', 'integer')
('reviews', 'review_id', 'integer')
('reviews', 'movie_id', 'integer')
('reviews', 'review_source', 'character varying')
('reviews', 'review_text', 'text')
('reviews', 'review_score', 'numeric')
('studios',

In [25]:
try:
    cur.execute("""
        DROP VIEW IF EXISTS schema_clean.vw_movies;

        CREATE VIEW schema_clean.vw_movies AS
        SELECT
            SUBSTRING(imdb_id FROM 3)::INTEGER AS movie_id,
            title,
            release_date,
            runtime
        FROM schema_clean.movies
        WHERE imdb_id ~ '^tt[0-9]{1,9}$';
    """)

    conn.commit()
    print("Movies view corrected!")

except Exception:
    conn.rollback()
    raise

Movies view corrected!


In [27]:
try:
    cur.execute("""
        CREATE OR REPLACE VIEW schema_clean.vw_movie_genres AS
        SELECT DISTINCT
            SUBSTRING(m.imdb_id FROM 3)::INTEGER AS movie_id,
            mg.genre_id
        FROM schema_clean.movie_genres mg
        JOIN schema_clean.movies m
            ON mg.movie_id = m.movie_id
        WHERE m.imdb_id ~ '^tt[0-9]{1,9}$';
    """)

    conn.commit()
    print("Movie genres view corrected!")

except Exception:
    conn.rollback()
    raise

Movie genres view corrected!


In [28]:
cur.execute("""
SELECT
    tc.table_name,
    tc.constraint_type,
    tc.constraint_name,
    string_agg(
        kcu.column_name,
        ', ' ORDER BY kcu.ordinal_position
    ) AS columns
FROM information_schema.table_constraints tc
LEFT JOIN information_schema.key_column_usage kcu
    ON tc.constraint_name = kcu.constraint_name
   AND tc.table_schema = kcu.table_schema
   AND tc.table_name = kcu.table_name
WHERE tc.table_schema = 'public'
  AND tc.table_name IN (
      'movies', 'genres', 'movie_genres',
      'people', 'movie_cast', 'movie_crew',
      'ratings'
  )
  AND tc.constraint_type IN (
      'PRIMARY KEY', 'FOREIGN KEY', 'UNIQUE'
  )
GROUP BY
    tc.table_name,
    tc.constraint_type,
    tc.constraint_name
ORDER BY tc.table_name, tc.constraint_type;
""")

for row in cur.fetchall():
    print(row)

conn.rollback()

('genres', 'PRIMARY KEY', 'genres_pkey', 'genre_id')
('genres', 'UNIQUE', 'genres_genre_name_key', 'genre_name')
('movie_cast', 'FOREIGN KEY', 'movie_cast_movie_id_fkey', 'movie_id')
('movie_cast', 'FOREIGN KEY', 'movie_cast_person_id_fkey', 'person_id')
('movie_cast', 'PRIMARY KEY', 'movie_cast_pkey', 'movie_id, person_id')
('movie_crew', 'FOREIGN KEY', 'movie_crew_movie_id_fkey', 'movie_id')
('movie_crew', 'FOREIGN KEY', 'movie_crew_person_id_fkey', 'person_id')
('movie_crew', 'PRIMARY KEY', 'movie_crew_pkey', 'movie_id, person_id, job')
('movie_genres', 'FOREIGN KEY', 'movie_genres_genre_id_fkey', 'genre_id')
('movie_genres', 'FOREIGN KEY', 'movie_genres_movie_id_fkey', 'movie_id')
('movie_genres', 'PRIMARY KEY', 'movie_genres_pkey', 'movie_id, genre_id')
('movies', 'PRIMARY KEY', 'movies_pkey', 'movie_id')
('people', 'PRIMARY KEY', 'people_pkey', 'person_id')
('ratings', 'FOREIGN KEY', 'ratings_movie_id_fkey', 'movie_id')
('ratings', 'PRIMARY KEY', 'ratings_pkey', 'rating_id')
('ra

In [29]:
conn.rollback()